<a href="https://colab.research.google.com/github/joaomaiavn02-lgtm/luxo-de-Bigdata-para-carga-de-dados-em-um-BD-NoSQL./blob/main/luxo%20de%20Bigdata%20para%20carga%20de%20dados%20em%20um%20BD%20NoSQL..ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Etapa 1 - Instalar bibliotecas

In [ ]:
!pip install pymongo pandas

Etapa 2 - Upload do CSV

In [ ]:
from google.colab import files

uploaded = files.upload()

Etapa 3 - Ler os dados

In [ ]:
import pandas as pd

df = pd.read_csv("netflix_titles.csv")

print("Total de registros:", len(df))

df.head()

Etapa 4 - Limpeza

In [ ]:
df.fillna("", inplace=True)

df.columns = df.columns.str.strip()

Etapa 5 - Transformação

director → diretores cast → elenco country → paises listed_in → categorias

seriam convertidos para listas.

In [ ]:
def criar_lista(valor):

    if valor == "":
        return []

    return [
        {"nome": item.strip()}
        for item in valor.split(",")
    ]

In [ ]:
df["diretores"] = df["director"].apply(criar_lista)
df["elenco"] = df["cast"].apply(criar_lista)
df["paises"] = df["country"].apply(criar_lista)
df["categorias"] = df["listed_in"].apply(criar_lista)

Etapa 6 - Transformar duração

Conforme seu modelo de agregados.

In [ ]:
def converter_duracao(valor):

    if valor == "":
        return {}

    partes = valor.split()

    return {
        "valor": partes[0],
        "unidade": " ".join(partes[1:])
    }

df["duracao"] = df["duration"].apply(converter_duracao)

Etapa 7 - Gerar documento MongoDB

In [ ]:
documentos = []

for _, row in df.iterrows():

    doc = {
        "show_id": row["show_id"],
        "tipo": row["type"],
        "titulo": row["title"],
        "diretores": row["diretores"],
        "elenco": row["elenco"],
        "paises": row["paises"],
        "data_adicionado": row["date_added"],
        "ano_lancamento": int(row["release_year"]),
        "classificacao": row["rating"],
        "duracao": row["duracao"],
        "categorias": row["categorias"],
        "descricao": row["description"]
    }

    documentos.append(doc)

Etapa 8 - Salvar JSON intermediário

In [ ]:
import json

with open("netflix_transformado.json", "w", encoding="utf-8") as f:
    json.dump(
        documentos,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )

Etapa 9 - Carga MongoDB

In [ ]:
from pymongo import MongoClient

client = MongoClient("SUA_STRING_ATLAS")

db = client["bigdata_netflix"]

colecao = db["netflix_shows"]

In [ ]:
colecao.delete_many({})

resultado = colecao.insert_many(documentos)

print(
    f"{len(resultado.inserted_ids)} documentos inseridos."
)

Etapa 10 - Consultas

As análises previstas do trabalho são: tipos de conteúdo, categorias, países, classificação indicativa e evolução do catálogo.

Filmes x Séries

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$tipo",
            "total": {"$sum": 1}
        }
    }
]

list(colecao.aggregate(pipeline))

Top Categorias

In [ ]:
pipeline = [
    {"$unwind": "$categorias"},
    {
        "$group": {
            "_id": "$categorias.nome",
            "total": {"$sum": 1}
        }
    },
    {"$sort": {"total": -1}},
    {"$limit": 10}
]

Top Países

In [ ]:
pipeline = [
    {"$unwind": "$paises"},
    {
        "$group": {
            "_id": "$paises.nome",
            "total": {"$sum": 1}
        }
    },
    {"$sort": {"total": -1}},
    {"$limit": 10}
]